# Bird Species Identification (CUB-200-2011)
UE24CS352A ML Mini-Project, Team A-29

**Kaggle setup (right sidebar):** Accelerator = GPU T4/P100 · Internet = ON (needed to download pretrained weights) · Add Data → search "CUB-200-2011" / "birds 200" and add any version. The loader detects the layout (original CUB files, or one folder per class) automatically.

Pipeline: data → classical baseline → ResNet transfer learning (phase 1: new head, phase 2: fine-tune) → evaluation → demo.

In [ ]:
import os, json, time, random, glob
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T

# ------------------------------ CONFIG ------------------------------
NUM_CLASSES = 0          # 0 = all 200 species; use 20 for a quick test run
USE_BBOX    = False      # True = crop to annotated bird box (only if the dataset has boxes)
ARCH        = "resnet50" # or "resnet18"
SEED        = 42
BATCH       = 32
BASE_TRAIN_PER_CLASS = 0 # >0 = baseline uses only this many train images per class (faster)
EPOCHS_P1   = 5          # phase 1: train new head only
EPOCHS_P2   = 12         # phase 2: fine-tune layer4 + head
LR_HEAD, LR_BACKBONE = 1e-3, 1e-4
OUT = Path("/kaggle/working") if Path("/kaggle").exists() else Path("outputs")
OUT.mkdir(exist_ok=True)

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| torch", torch.__version__)

## 1. Data
Find the dataset folder, read the CUB metadata files, pick classes, make train / val / test splits.

In [ ]:
IMG_EXT = (".jpg", ".jpeg")
BASE = "/kaggle/input" if Path("/kaggle/input").exists() else "."
hits = glob.glob(f"{BASE}/**/images.txt", recursive=True)

if hits and (Path(hits[0]).parent / "images").exists():
    # ---- layout A: original CUB-200-2011 metadata files ----
    ROOT = Path(hits[0]).parent
    def read_pairs(name):
        return [l.split(maxsplit=1) for l in (ROOT / name).read_text().splitlines() if l.strip()]
    paths   = {int(i): p for i, p in read_pairs("images.txt")}
    labels  = {int(i): int(c) for i, c in read_pairs("image_class_labels.txt")}
    split   = {int(i): int(t) for i, t in read_pairs("train_test_split.txt")}
    bbox    = {int(i): tuple(map(float, b.split())) for i, b in read_pairs("bounding_boxes.txt")}
    classes = {int(i): n for i, n in read_pairs("classes.txt")}
    df = pd.DataFrame({"id": list(paths), "path": [str(ROOT / "images" / paths[i]) for i in paths]})
    df["cls"] = df.id.map(labels); df["is_train"] = df.id.map(split); df["bbox"] = df.id.map(bbox)
    HAS_BBOX = True
    print("layout A (original CUB files):", ROOT)
else:
    # ---- layout B: one folder per class (modified CUB versions on Kaggle) ----
    files = sorted(Path(p) for p in glob.glob(f"{BASE}/**/*", recursive=True) if p.lower().endswith(IMG_EXT))
    assert files, "no .jpg images found - check 'Add Data' (and share the folder tree if it persists)"
    def split_of(p):
        parts = {x.lower() for x in p.parts}
        if parts & {"train", "training"}: return 1
        if parts & {"test", "testing"}: return 0
        if parts & {"val", "valid", "validation"}: return 2
        return -1
    df = pd.DataFrame({"id": range(len(files)), "path": [str(p) for p in files],
                       "cls": [p.parent.name for p in files], "is_train": [split_of(p) for p in files]})
    df["bbox"] = None; HAS_BBOX = False
    classes = {c: c for c in df.cls.unique()}
    print(f"layout B (folder per class): {len(files)} images, split labels {df.is_train.value_counts().to_dict()}")
if USE_BBOX and not HAS_BBOX:
    print("no bounding boxes in this version -> USE_BBOX=False"); USE_BBOX = False

import re
ids = sorted(df.cls.unique())
chosen = ids if NUM_CLASSES <= 0 else sorted(np.random.default_rng(SEED).choice(ids, NUM_CLASSES, replace=False).tolist())
df = df[df.cls.isin(chosen)].reset_index(drop=True)
df["y"] = df.cls.map({c: i for i, c in enumerate(chosen)})
names = [re.sub(r"^\d+\.?", "", str(classes[c])).replace("_", " ").strip() or str(c) for c in chosen]

from sklearn.model_selection import train_test_split
if (df.is_train == 1).any() and (df.is_train == 0).any():          # provided train/test split
    train_all = df[df.is_train == 1]; test_df = df[df.is_train == 0].reset_index(drop=True)
    given_val = df[df.is_train == 2]
elif (df.is_train == 1).any() and (df.is_train == 2).any():        # only train + valid folders
    train_all = df[df.is_train == 1]; test_df = df[df.is_train == 2].reset_index(drop=True)
    given_val = df.iloc[0:0]
else:                                                              # no split info: 50/50 stratified
    train_all, test_df = train_test_split(df, test_size=0.5, stratify=df.y, random_state=SEED)
    test_df = test_df.reset_index(drop=True); given_val = df.iloc[0:0]
# the test set is only used for the final report; a validation set picks the best epoch
if len(given_val):
    train_df, val_df = train_all, given_val
else:
    train_df, val_df = train_test_split(train_all, test_size=0.15, stratify=train_all.y, random_state=SEED)
train_df, val_df = train_df.reset_index(drop=True), val_df.reset_index(drop=True)
print(f"{len(chosen)} classes | train {len(train_df)} | val {len(val_df)} | test {len(test_df)}")

## 2. Classical baseline
Hand-made features (HSV colour histogram + HOG + LBP) fed to classical classifiers. Same train/test images as the CNN.

In [ ]:
from joblib import Parallel, delayed
from skimage.feature import hog, local_binary_pattern
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

def topk_acc(s, y, k=5):
    k = min(k, s.shape[1]); top = np.argsort(-s, axis=1)[:, :k]
    return float((top == y[:, None]).any(1).mean())

def scores(clf, X):
    return clf.predict_proba(X) if hasattr(clf, "predict_proba") else clf.decision_function(X)

def hand_features(path, bb):
    img = Image.open(path)
    if not USE_BBOX: img.draft("RGB", (256, 256))   # decode big JPEGs at reduced size (faster)
    img = img.convert("RGB")
    if USE_BBOX:
        bx, by, bw, bh = bb; img = img.crop((int(bx), int(by), int(bx + bw), int(by + bh)))
    img = img.resize((128, 128))
    hsv = np.asarray(img.convert("HSV"))
    hist, _ = np.histogramdd(hsv.reshape(-1, 3), bins=(8, 8, 8), range=((0, 256),) * 3)
    hist = hist.ravel() / hist.sum()
    gray = np.asarray(img.convert("L"))
    h = hog(gray, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2))
    lbp = local_binary_pattern(gray, P=8, R=1, method="uniform")
    lh, _ = np.histogram(lbp, bins=10, range=(0, 10), density=True)
    return np.concatenate([hist, h, lh]).astype(np.float32)

from tqdm.auto import tqdm
N_JOBS = min(4, os.cpu_count() or 1)      # n_jobs=-1 can over-spawn workers inside Kaggle's container

btr = train_df if not BASE_TRAIN_PER_CLASS else train_df.groupby("y", group_keys=False).head(BASE_TRAIN_PER_CLASS)

# quick timing check on 20 images before the full run
t = time.time()
for p, b in zip(btr.path[:20], btr.bbox[:20]): hand_features(p, b)
per = (time.time() - t) / 20
print(f"{per * 1000:.0f} ms/image -> ~{per * (len(btr) + len(test_df)) / N_JOBS / 60:.1f} min estimated with {N_JOBS} workers")

def feats(d):
    gen = Parallel(n_jobs=N_JOBS, return_as="generator")(delayed(hand_features)(p, b) for p, b in zip(d.path, d.bbox))
    return np.stack([f for f in tqdm(gen, total=len(d))])

t = time.time(); Xtr, Xte = feats(btr), feats(test_df)
ytr, yte = btr.y.values, test_df.y.values
print(f"features {Xtr.shape[1]}-d extracted in {time.time() - t:.0f}s")

NPC = min(256, Xtr.shape[0] - 1, Xtr.shape[1])     # PCA size for the linear SVM (smaller when few images)
base_models = {
    "LinearSVM":    make_pipeline(StandardScaler(), PCA(n_components=NPC, random_state=SEED),
                                  LinearSVC(C=0.01, max_iter=1000, random_state=SEED)),
    "RBF-SVM":      make_pipeline(StandardScaler(), PCA(n_components=0.95, random_state=SEED), SVC(C=10, gamma="scale")),
    "RandomForest": RandomForestClassifier(n_estimators=300, n_jobs=N_JOBS, random_state=SEED),
    "kNN":          make_pipeline(StandardScaler(), PCA(n_components=0.95, random_state=SEED),
                                  KNeighborsClassifier(n_neighbors=5, weights="distance")),
}
base_res = {}
for name, clf in base_models.items():
    print(f"fitting {name} ...", flush=True)
    t = time.time(); clf.fit(Xtr, ytr); s = scores(clf, Xte)
    base_res[name] = {"top1": float((s.argmax(1) == yte).mean()), "top5": topk_acc(s, yte)}
    print(f"{name:13s} top1={base_res[name]['top1']:.3f} top5={base_res[name]['top5']:.3f} ({time.time() - t:.0f}s)")
best_base = max(base_res, key=lambda k: base_res[k]["top1"])
print("best baseline:", best_base)

## 3. CNN transfer learning
- Pretrained ResNet (ImageNet) with its last layer replaced by a new `Linear(features, N species)`.
- **Phase 1:** freeze the backbone, train only the new head.
- **Phase 2:** also unfreeze `layer4`, train with a lower learning rate.
- Frozen BatchNorm layers stay in eval mode so their ImageNet statistics are not disturbed.

In [ ]:
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # ImageNet statistics
train_tf = T.Compose([T.RandomResizedCrop(224, scale=(0.6, 1.0)), T.RandomHorizontalFlip(),
                      T.ColorJitter(0.2, 0.2, 0.2), T.ToTensor(), T.Normalize(mean, std)])
eval_tf  = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(mean, std)])

class Birds(Dataset):
    def __init__(self, d, tf): self.d, self.tf = d.reset_index(drop=True), tf
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        r = self.d.iloc[i]
        img = Image.open(r.path).convert("RGB")
        if USE_BBOX:
            bx, by, bw, bh = r.bbox; img = img.crop((int(bx), int(by), int(bx + bw), int(by + bh)))
        return self.tf(img), int(r.y)

def loader(d, tf, shuffle):
    return DataLoader(Birds(d, tf), batch_size=BATCH, shuffle=shuffle, num_workers=2,
                      pin_memory=(device == "cuda"))
train_dl, val_dl, test_dl = loader(train_df, train_tf, True), loader(val_df, eval_tf, False), loader(test_df, eval_tf, False)

def build_model():
    m = models.get_model(ARCH, weights="DEFAULT")      # pretrained on ImageNet
    m.fc = nn.Linear(m.fc.in_features, len(chosen))    # new classifier head
    return m.to(device)

def set_trainable(m, blocks):
    for n, p in m.named_parameters():
        p.requires_grad = n.startswith("fc") or any(n.startswith(b) for b in blocks)

def set_train_mode(m):
    m.train()
    for mod in m.modules():                            # keep frozen BatchNorms in eval mode
        if isinstance(mod, nn.BatchNorm2d) and not any(p.requires_grad for p in mod.parameters()):
            mod.eval()

@torch.no_grad()
def predict(m, dl):
    m.eval(); outs, ys = [], []
    for x, y in dl:
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=(device == "cuda")):
            outs.append(m(x.to(device)).float().cpu())
        ys.append(y)
    return torch.cat(outs).numpy(), torch.cat(ys).numpy()

def run_phase(m, name, epochs, blocks, lr_backbone):
    set_trainable(m, blocks)
    head = [p for n, p in m.named_parameters() if p.requires_grad and n.startswith("fc")]
    body = [p for n, p in m.named_parameters() if p.requires_grad and not n.startswith("fc")]
    groups = [{"params": head, "lr": LR_HEAD}] + ([{"params": body, "lr": lr_backbone}] if body else [])
    opt = torch.optim.AdamW(groups, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
    print(f"[{name}] trainable params: {sum(p.numel() for p in head + body):,}")
    best, best_state, hist = -1.0, None, []
    for ep in range(epochs):
        set_train_mode(m); t = time.time(); tot = corr = n = 0
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=(device == "cuda")):
                out = m(x); loss = crit(out, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item() * len(y); corr += (out.argmax(1) == y).sum().item(); n += len(y)
        sched.step()
        sv, yv = predict(m, val_dl); vacc = float((sv.argmax(1) == yv).mean())
        hist.append({"phase": name, "epoch": ep + 1, "train_loss": tot / n, "train_acc": corr / n, "val_acc": vacc})
        print(f"[{name}] ep {ep + 1:2d}/{epochs}  loss {tot / n:.3f}  train {corr / n:.3f}  val {vacc:.3f}  ({time.time() - t:.0f}s)")
        if vacc > best:
            best = vacc; best_state = {k: v.detach().clone() for k, v in m.state_dict().items()}
    m.load_state_dict(best_state)                       # restore best epoch (by validation)
    return hist

In [ ]:
model = build_model()

hist = run_phase(model, "phase1", EPOCHS_P1, [], None)
s1, y1 = predict(model, test_dl)
p1 = {"top1": float((s1.argmax(1) == y1).mean()), "top5": topk_acc(s1, y1)}
print("after phase 1 (test):", p1)

hist += run_phase(model, "phase2", EPOCHS_P2, ["layer4"], LR_BACKBONE)
s2, y2 = predict(model, test_dl)
p2 = {"top1": float((s2.argmax(1) == y2).mean()), "top5": topk_acc(s2, y2)}
print("after phase 2 (test):", p2)

## 4. Evaluation

In [ ]:
summary = pd.DataFrame({
    f"Baseline ({best_base})": base_res[best_base],
    "CNN phase 1 (frozen backbone)": p1,
    "CNN phase 2 (fine-tuned)": p2,
}).T.round(4)
display(summary)

# training curves
h = pd.DataFrame(hist); h["step"] = range(1, len(h) + 1)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(h.step, h.train_loss); ax[0].set_title("train loss"); ax[0].set_xlabel("epoch (phase 1 then 2)")
ax[1].plot(h.step, h.train_acc, label="train"); ax[1].plot(h.step, h.val_acc, label="val")
ax[1].set_title("accuracy"); ax[1].legend(); ax[1].set_xlabel("epoch (phase 1 then 2)")
for a in ax: a.axvline(EPOCHS_P1 + 0.5, ls="--", c="gray")
plt.tight_layout(); plt.savefig(OUT / "training_curves.png", dpi=150); plt.show()

# confusion matrix + most confused / hardest species
from sklearn.metrics import confusion_matrix
pred = s2.argmax(1)
cm = confusion_matrix(y2, pred, labels=range(len(chosen)))
fig, ax = plt.subplots(figsize=(7, 6)); im = ax.imshow(cm, cmap="Blues"); fig.colorbar(im)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Confusion matrix (CNN phase 2)")
if len(chosen) <= 25:
    ax.set_xticks(range(len(names)), names, rotation=90, fontsize=6); ax.set_yticks(range(len(names)), names, fontsize=6)
plt.tight_layout(); plt.savefig(OUT / "confusion_matrix.png", dpi=150); plt.show()

off = cm.copy(); np.fill_diagonal(off, 0)
print("\nMost confused pairs (true -> predicted, count):")
for a, b in np.dstack(np.unravel_index(np.argsort(-off.ravel())[:8], off.shape))[0]:
    if off[a, b] > 0: print(f"  {names[a]}  ->  {names[b]}  ({off[a, b]})")
per_class = cm.diagonal() / cm.sum(1).clip(min=1)
print("\nHardest species:", [(names[i], round(float(per_class[i]), 2)) for i in np.argsort(per_class)[:8]])

results = {"config": dict(NUM_CLASSES=len(chosen), USE_BBOX=USE_BBOX, ARCH=ARCH, EPOCHS_P1=EPOCHS_P1, EPOCHS_P2=EPOCHS_P2),
           "baseline": base_res, "cnn_phase1": p1, "cnn_phase2": p2}
(OUT / "results.json").write_text(json.dumps(results, indent=2))
torch.save({"state_dict": model.state_dict(), "arch": ARCH, "names": names, "use_bbox": USE_BBOX}, OUT / "bird_model.pth")
print("\nsaved results.json, bird_model.pth, figures to", OUT)

## 5. Demo
Top-3 predictions with confidence. To try your own photo: upload it to the notebook (Add Data → Upload) and call `predict_image('/kaggle/input/<your-dataset>/photo.jpg')`.

In [ ]:
def predict_image(src, topk=3, title=None):
    img = Image.open(src).convert("RGB")
    x = eval_tf(img).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        p = model(x).float().softmax(1)[0].cpu()
    conf, idx = p.topk(topk)
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.4), gridspec_kw={"width_ratios": [1, 1.3]})
    ax[0].imshow(img); ax[0].axis("off"); ax[0].set_title(title or "input")
    ax[1].barh([names[i] for i in idx][::-1], conf.numpy()[::-1]); ax[1].set_xlim(0, 1); ax[1].set_xlabel("confidence")
    plt.tight_layout(); plt.show()
    return [(names[i], round(float(c), 3)) for i, c in zip(idx, conf)]

for i in np.random.default_rng(0).choice(len(test_df), 4, replace=False):
    r = test_df.iloc[i]
    print(predict_image(r.path, title=f"true: {names[r.y]}"))